In [1]:
# ═══════════════════════════════════════════════════════════════════
# ML WAREHOUSE OPERATIONS — INTERACTIVE DECISION APP
# By: Garvit Mittal
# ═══════════════════════════════════════════════════════════════════

!pip install prophet xgboost scikit-learn ipywidgets --quiet

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✅ All libraries loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 3.0 MB/s eta 0:00:00
✅ All libraries loaded!


In [3]:
from google.colab import files

print("Upload your processed CSV files:")
print("Required: shipments.csv, shifts.csv, throughput_scans.csv, disruptions.csv, dock_events.csv")
uploaded = files.upload()
print("✅ Files uploaded!")

# Load all dataframes
import pandas as pd
shipments    = pd.read_csv('shipments.csv')
shifts       = pd.read_csv('shifts.csv')
throughput   = pd.read_csv('throughput_scans.csv')
disruptions  = pd.read_csv('disruptions.csv')
dock_events  = pd.read_csv('dock_events.csv')

print(f"\n✅ All files loaded!")
print(f"   Shipments:    {len(shipments)} rows")
print(f"   Shifts:       {len(shifts)} rows")
print(f"   Throughput:   {len(throughput)} rows")
print(f"   Disruptions:  {len(disruptions)} rows")
print(f"   Dock Events:  {len(dock_events)} rows")

Upload your processed CSV files:
Required: shipments.csv, shifts.csv, throughput_scans.csv, disruptions.csv, dock_events.csv


Saving disruptions.csv to disruptions (1).csv
Saving dock_events.csv to dock_events.csv
Saving shifts.csv to shifts (1).csv
Saving shipments.csv to shipments (1).csv
Saving throughput_scans.csv to throughput_scans (1).csv
✅ Files uploaded!

✅ All files loaded!
   Shipments:    180519 rows
   Shifts:       3381 rows
   Throughput:   27048 rows
   Disruptions:  131 rows
   Dock Events:  172765 rows


In [8]:
# ═══════════════════════════════════════════════════════════════════
# TRAIN ALL MODELS
# ═══════════════════════════════════════════════════════════════════
from prophet import Prophet
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Train Prophet demand model ────────────────────────────────────
print("🔄 Training Demand Forecast model...")
daily = shipments.groupby('shipment_date').size().reset_index()
daily.columns = ['ds', 'y']
daily['ds'] = pd.to_datetime(daily['ds'])
daily = daily.sort_values('ds')

prophet_model = Prophet(yearly_seasonality=True,
                        weekly_seasonality=True,
                        daily_seasonality=False)
prophet_model.fit(daily)
print("✅ Prophet model trained!")

# ── Train Throughput model ────────────────────────────────────────
print("🔄 Training Throughput Predictor model...")
th = throughput.copy()
th['scan_datetime'] = pd.to_datetime(th['scan_datetime'])
th['hour']       = th['scan_datetime'].dt.hour
th['is_weekend'] = th['scan_datetime'].dt.dayofweek >= 5
th['month']      = th['scan_datetime'].dt.month

th = th.merge(shifts[['shift_id','shift_type','actual_workers']],
              on='shift_id', how='left')
th = th.dropna(subset=['shift_type'])

le_shift    = LabelEncoder()
le_scantype = LabelEncoder()
th['shift_type_enc'] = le_shift.fit_transform(th['shift_type'])
th['scan_type_enc']  = le_scantype.fit_transform(th['scan_type'])

features = ['actual_workers','shift_type_enc','hour',
            'scan_type_enc','is_weekend','month']
X = th[features]
y = th['units_scanned']

gb_model = GradientBoostingRegressor(n_estimators=100,
                                      learning_rate=0.1,
                                      max_depth=4,
                                      random_state=42)
gb_model.fit(X, y)
print("✅ Throughput model trained!")

# ── Staffing recommendation function ─────────────────────────────
def recommend_staff(forecasted_shipments):
    units = forecasted_shipments * 2.2
    per_shift = units / 3
    def calc_shift(u):
        staff = max(4, int(u / 50))
        return {
            'recommended_staff': staff,
            'pickers_sorters':   max(1, int(staff * 0.5)),
            'loaders':           max(1, int(staff * 0.25)),
            'forklift_ops':      max(1, int(staff * 0.25)),
        }
    return {
        'morning':   calc_shift(per_shift),
        'afternoon': calc_shift(per_shift * 0.8),
        'night':     calc_shift(per_shift * 0.5),
    }

# ── Dock events summary ───────────────────────────────────────────
print("🔄 Preparing Dock Events data...")
dock_events['event_date'] = pd.to_datetime(dock_events['event_date'])
dock_events['hour']      = pd.to_datetime(dock_events['scheduled_start']).dt.hour
dock_events['day_name']  = dock_events['event_date'].dt.day_name()
print("✅ Dock Events ready!")

print("\n🎉 All models ready! Proceed to Cell 4 for the interactive app.")

🔄 Training Demand Forecast model...
✅ Prophet model trained!
🔄 Training Throughput Predictor model...
✅ Throughput model trained!
🔄 Preparing Dock Events data...
✅ Dock Events ready!

🎉 All models ready! Proceed to Cell 4 for the interactive app.


In [16]:
# ═══════════════════════════════════════════════════════════════════
# 🏭 INTERACTIVE WAREHOUSE DECISION APP
# ═══════════════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

display(HTML("""
<div style='background:#1a237e;padding:20px;border-radius:10px;margin-bottom:20px'>
  <h2 style='color:white;margin:0'>🏭 ML Warehouse Operations Decision System</h2>
  <p style='color:#90caf9;margin:5px 0 0'>By Garvit Mittal | Real-time ML Predictions</p>
</div>
"""))

# ── Widgets ───────────────────────────────────────────────────────
forecast_days = widgets.IntSlider(
    value=7, min=1, max=30, step=1,
    description='Forecast Days:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)
workers_input = widgets.IntSlider(
    value=8, min=1, max=30, step=1,
    description='Workers on Shift:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)
shift_input = widgets.Dropdown(
    options=['morning', 'afternoon', 'night'],
    value='morning',
    description='Shift Type:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)
hour_input = widgets.IntSlider(
    value=9, min=0, max=23, step=1,
    description='Hour of Day:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)
scan_type_input = widgets.Dropdown(
    options=list(le_scantype.classes_),
    description='Scan Type:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)
weekend_input = widgets.Checkbox(
    value=False,
    description='Is Weekend?',
    style={'description_width': '150px'}
)
dock_door_input = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='Dock Door #:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)
run_button = widgets.Button(
    description='🚀 Run Predictions',
    button_style='success',
    layout=widgets.Layout(width='220px', height='45px')
)
output = widgets.Output()

# ── Display widgets ───────────────────────────────────────────────
display(HTML("<h3 style='color:#64b5f6'>📅 Demand Forecast Settings</h3>"))
display(forecast_days)

display(HTML("<h3 style='color:#64b5f6'>⚙️ Throughput Prediction Settings</h3>"))
display(workers_input, shift_input, hour_input, scan_type_input, weekend_input)

display(HTML("<h3 style='color:#64b5f6'>🚚 Dock Settings</h3>"))
display(dock_door_input)

display(run_button)
display(output)

# ── Button logic ──────────────────────────────────────────────────
def on_run_clicked(b):
    with output:
        clear_output()

        # 1. Demand Forecast
        future    = prophet_model.make_future_dataframe(periods=forecast_days.value, freq='D')
        forecast  = prophet_model.predict(future)
        future_fc = forecast.tail(forecast_days.value)[['ds','yhat','yhat_lower','yhat_upper']]
        avg_shipments = future_fc['yhat'].mean()

        # 2. Staffing
        rec = recommend_staff(avg_shipments)

        # 3. Throughput
        shift_enc = le_shift.transform([shift_input.value.lower()])[0]
        scan_enc  = le_scantype.transform([scan_type_input.value])[0]
        X_pred    = pd.DataFrame([[
            workers_input.value, shift_enc,
            hour_input.value, scan_enc,
            int(weekend_input.value),
            pd.Timestamp.now().month
        ]], columns=['actual_workers','shift_type_enc','hour',
                     'scan_type_enc','is_weekend','month'])
        predicted_units = gb_model.predict(X_pred)[0]

        # 4. Dock analysis
        dock_filtered = dock_events[dock_events['dock_door'] == dock_door_input.value]
        avg_wait      = dock_filtered['wait_time_min'].mean() if len(dock_filtered) > 0 else 0
        total_events  = len(dock_filtered)
        busiest_day   = dock_filtered['day_name'].mode()[0] if len(dock_filtered) > 0 else 'N/A'

        # ── Display results ───────────────────────────────────────
        display(HTML(f"""
        <div style='background:#e8f5e9;padding:15px;border-radius:8px;margin:10px 0;
                    border-left:5px solid #4caf50'>
          <h3 style='color:#2e7d32;margin:0'>📈 DEMAND FORECAST (Next {forecast_days.value} days)</h3>
          <p style='font-size:16px;margin:8px 0;color:#000000'>
            Avg Daily Shipments: <b>{avg_shipments:.0f}</b><br>
            Range: {future_fc['yhat_lower'].mean():.0f} –
                   {future_fc['yhat_upper'].mean():.0f} shipments/day
          </p>
        </div>

        <div style='background:#e3f2fd;padding:15px;border-radius:8px;margin:10px 0;
                    border-left:5px solid #2196f3'>
          <h3 style='color:#1565c0;margin:0'>👷 STAFFING RECOMMENDATION</h3>
          <p style='font-size:16px;margin:8px 0;color:#000000'>
            Based on {avg_shipments:.0f} avg daily shipments:<br>
            🌅 Morning:   <b>{rec['morning']['recommended_staff']} workers</b>
               (Pickers: {rec['morning']['pickers_sorters']},
                Loaders: {rec['morning']['loaders']})<br>
            ☀️ Afternoon: <b>{rec['afternoon']['recommended_staff']} workers</b>
               (Pickers: {rec['afternoon']['pickers_sorters']},
                Loaders: {rec['afternoon']['loaders']})<br>
            🌙 Night:     <b>{rec['night']['recommended_staff']} workers</b>
               (Pickers: {rec['night']['pickers_sorters']},
                Loaders: {rec['night']['loaders']})
          </p>
        </div>

        <div style='background:#fff3e0;padding:15px;border-radius:8px;margin:10px 0;
                    border-left:5px solid #ff9800'>
          <h3 style='color:#e65100;margin:0'>📦 THROUGHPUT PREDICTION</h3>
          <p style='font-size:16px;margin:8px 0;color:#000000'>
            {shift_input.value.title()} shift | {workers_input.value} workers |
            Hour {hour_input.value:02d}:00<br>
            Predicted Units Scanned: <b>{predicted_units:.0f} units</b>
          </p>
        </div>

        <div style='background:#f3e5f5;padding:15px;border-radius:8px;margin:10px 0;
                    border-left:5px solid #9c27b0'>
          <h3 style='color:#6a1b9a;margin:0'>🚚 DOCK DOOR {dock_door_input.value} ANALYSIS</h3>
          <p style='font-size:16px;margin:8px 0;color:#000000'>
            Total Events:  <b>{total_events}</b><br>
            Avg Wait Time: <b>{avg_wait:.1f} minutes</b><br>
            Busiest Day:   <b>{busiest_day}</b>
          </p>
        </div>

        <p style='color:#aaaaaa;font-size:12px'>✅ Predictions generated successfully</p>
        """))

run_button.on_click(on_run_clicked)

IntSlider(value=7, description='Forecast Days:', layout=Layout(width='500px'), max=30, min=1, style=SliderStyl…

IntSlider(value=8, description='Workers on Shift:', layout=Layout(width='500px'), max=30, min=1, style=SliderS…

Dropdown(description='Shift Type:', layout=Layout(width='500px'), options=('morning', 'afternoon', 'night'), s…

IntSlider(value=9, description='Hour of Day:', layout=Layout(width='500px'), max=23, style=SliderStyle(descrip…

Dropdown(description='Scan Type:', layout=Layout(width='500px'), options=('dispatch', 'load', 'receive', 'sort…

Checkbox(value=False, description='Is Weekend?', style=DescriptionStyle(description_width='150px'))

IntSlider(value=3, description='Dock Door #:', layout=Layout(width='500px'), max=10, min=1, style=SliderStyle(…

Button(button_style='success', description='🚀 Run Predictions', layout=Layout(height='45px', width='220px'), s…

Output()